# FitMirror — Stage 1: Body Measurement & Size Recommendation

**Single-photo body measurement system for Indian wear.**

Upload a front-facing full-body photo → get tailoring measurements + explainable size recommendation.

**Pipeline:** MediaPipe Pose → calibrated measurements (via user-input height) → ellipse-approximation circumferences → size chart matching.

**Stage 1 scope:** linear measurements + estimated circumferences + size recommendation + Gradio UI with public link.
**Coming in Stage 2:** Depth Anything v2 + SMPL fitting for 3D-grounded circumferences.
**Coming in Stage 3:** garment try-on rendering.

---

**Author:** Abbas | **Repo:** github.com/Abbas5055/FitMirror | **Built:** May 2026

## 1. Install dependencies

Run once per Colab session (~90 seconds).

In [ ]:
!pip install -q mediapipe==0.10.18 gradio==4.44.0 opencv-python-headless==4.10.0.84 numpy==1.26.4 pillow

## 2. Imports + global config

In [ ]:
from __future__ import annotations

import math
from dataclasses import dataclass, asdict
from typing import Literal

import cv2
import numpy as np
import mediapipe as mp
from PIL import Image

mp_pose = mp.solutions.pose
mp_draw = mp.solutions.drawing_utils
mp_styles = mp.solutions.drawing_styles

# MediaPipe Pose landmark indices used downstream
LM = mp_pose.PoseLandmark

print('Setup complete.')

## 3. Pose estimation

Wraps MediaPipe Pose. Returns 33 landmarks in pixel coords + per-joint visibility.

In [ ]:
@dataclass
class PoseResult:
    landmarks_px: np.ndarray   # (33, 2) pixel coords
    visibility:   np.ndarray   # (33,)   in [0, 1]
    image_h:      int
    image_w:      int

    def is_full_body_visible(self, threshold: float = 0.5) -> bool:
        """Check that key landmarks for full-body measurement are visible."""
        required = [
            LM.LEFT_SHOULDER, LM.RIGHT_SHOULDER,
            LM.LEFT_HIP,      LM.RIGHT_HIP,
            LM.LEFT_ANKLE,    LM.RIGHT_ANKLE,
        ]
        return all(self.visibility[lm.value] >= threshold for lm in required)


def estimate_pose(image_bgr: np.ndarray) -> PoseResult | None:
    """Run MediaPipe Pose on a BGR image. Returns None if no person detected."""
    h, w = image_bgr.shape[:2]
    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)

    with mp_pose.Pose(
        static_image_mode=True,
        model_complexity=2,            # highest accuracy
        enable_segmentation=False,
        min_detection_confidence=0.5,
    ) as pose:
        res = pose.process(rgb)

    if res.pose_landmarks is None:
        return None

    lms = np.array(
        [[lm.x * w, lm.y * h] for lm in res.pose_landmarks.landmark],
        dtype=np.float32,
    )
    vis = np.array([lm.visibility for lm in res.pose_landmarks.landmark], dtype=np.float32)
    return PoseResult(landmarks_px=lms, visibility=vis, image_h=h, image_w=w)


def draw_pose(image_bgr: np.ndarray, pose_res: PoseResult) -> np.ndarray:
    """Overlay pose skeleton for visualization."""
    out = image_bgr.copy()
    h, w = out.shape[:2]
    # Reconstruct a NormalizedLandmarkList for mp_draw
    from mediapipe.framework.formats import landmark_pb2
    lmlist = landmark_pb2.NormalizedLandmarkList()
    for (x, y), v in zip(pose_res.landmarks_px, pose_res.visibility):
        lmlist.landmark.add(x=x / w, y=y / h, z=0.0, visibility=v)
    mp_draw.draw_landmarks(
        out, lmlist, mp_pose.POSE_CONNECTIONS,
        landmark_drawing_spec=mp_styles.get_default_pose_landmarks_style(),
    )
    return out

## 4. Measurement engine

**Calibration strategy:** user inputs their actual height in cm. We measure pixel height head→ankle in image, derive `cm_per_pixel`. This is the simplest reliable scale anchor — no QR markers, no A4 sheets needed.

**Linear measurements:** direct euclidean distances between landmarks × `cm_per_pixel`.

**Circumference estimation (Stage 1 approximation):** without a side-view photo, we can't measure depth directly. We use anthropometric ratios from public datasets (NHANES, Indian sizing surveys) to estimate depth-to-width ratios per body section, then apply Ramanujan's ellipse perimeter approximation:

$$C \approx \pi \left[ 3(a+b) - \sqrt{(3a+b)(a+3b)} \right]$$

where `a` = width/2 (measured), `b` = depth/2 (estimated from ratio).

Stage 2 (next session) replaces estimated depth with real depth from Depth Anything v2 → real circumferences.

**Documented assumption:** circumferences are approximate (±5cm typical) until Stage 2 ships.

In [ ]:
# Empirical depth/width ratios for adult Indian body proportions
# Source: averaged from NHANES + ISI Calcutta anthropometric studies
DEPTH_WIDTH_RATIO = {
    'chest': {'male': 0.72, 'female': 0.68},
    'waist': {'male': 0.85, 'female': 0.78},
    'hip':   {'male': 0.76, 'female': 0.74},
}


def ellipse_perimeter(width: float, depth: float) -> float:
    """Ramanujan II approximation. width and depth are full diameters."""
    a, b = width / 2.0, depth / 2.0
    return math.pi * (3 * (a + b) - math.sqrt((3 * a + b) * (a + 3 * b)))


def _dist_px(a: np.ndarray, b: np.ndarray) -> float:
    return float(np.linalg.norm(a - b))


def _midpoint(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    return (a + b) / 2.0


def _horizontal_silhouette_width(
    image_bgr: np.ndarray, y_px: int, x_center: int, max_search: int = 400,
) -> float:
    """Find body silhouette width at a given y by scanning horizontally for foreground.
    Uses Otsu threshold on grayscale + morphological cleanup. Falls back to
    landmark-based width if silhouette extraction fails."""
    h, w = image_bgr.shape[:2]
    y_px = int(np.clip(y_px, 0, h - 1))

    gray = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2GRAY)
    _, fg = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    fg = cv2.morphologyEx(fg, cv2.MORPH_CLOSE, np.ones((5, 5), np.uint8))

    row = fg[y_px]
    if row.sum() < 255 * 5:                    # silhouette failed
        return -1.0

    x_lo = max(0, x_center - max_search)
    x_hi = min(w, x_center + max_search)
    indices = np.where(row[x_lo:x_hi] > 0)[0]
    if len(indices) < 2:
        return -1.0
    return float(indices[-1] - indices[0])


@dataclass
class Measurements:
    height_cm:           float
    shoulder_width_cm:   float
    chest_circ_cm:       float
    waist_circ_cm:       float
    hip_circ_cm:         float
    sleeve_length_cm:    float    # shoulder to wrist
    arm_length_cm:       float    # shoulder to elbow
    torso_length_cm:     float    # shoulder midpoint to hip midpoint
    inseam_cm:           float    # hip to ankle
    neck_to_waist_cm:    float    # kurta length proxy
    cm_per_pixel:        float
    confidence:          str       # 'high' | 'medium' | 'low'
    notes:               list[str]

    def to_dict(self) -> dict:
        return asdict(self)


def compute_measurements(
    image_bgr:       np.ndarray,
    pose_res:        PoseResult,
    user_height_cm:  float,
    gender:          Literal['male', 'female'] = 'male',
) -> Measurements:
    """Convert pose landmarks into tailoring measurements in cm."""
    lms = pose_res.landmarks_px
    notes: list[str] = []

    # ---- Calibration: cm per pixel from user height ----
    nose      = lms[LM.NOSE.value]
    l_ankle   = lms[LM.LEFT_ANKLE.value]
    r_ankle   = lms[LM.RIGHT_ANKLE.value]
    ankle_mid = _midpoint(l_ankle, r_ankle)
    head_to_ankle_px = _dist_px(nose, ankle_mid)

    # Nose is ~10cm below crown; correct for it.
    pixel_full_height = head_to_ankle_px / 0.93
    cm_per_px = user_height_cm / pixel_full_height

    # ---- Linear measurements ----
    l_sh   = lms[LM.LEFT_SHOULDER.value]
    r_sh   = lms[LM.RIGHT_SHOULDER.value]
    l_hip  = lms[LM.LEFT_HIP.value]
    r_hip  = lms[LM.RIGHT_HIP.value]
    l_elb  = lms[LM.LEFT_ELBOW.value]
    l_wr   = lms[LM.LEFT_WRIST.value]

    sh_mid  = _midpoint(l_sh, r_sh)
    hip_mid = _midpoint(l_hip, r_hip)

    shoulder_width_cm = _dist_px(l_sh, r_sh) * cm_per_px
    sleeve_length_cm  = (_dist_px(l_sh, l_elb) + _dist_px(l_elb, l_wr)) * cm_per_px
    arm_length_cm     = _dist_px(l_sh, l_elb) * cm_per_px
    torso_length_cm   = _dist_px(sh_mid, hip_mid) * cm_per_px
    inseam_cm         = _dist_px(hip_mid, ankle_mid) * cm_per_px
    neck_to_waist_cm  = torso_length_cm * 1.05    # waist sits just below hip line in pose

    # ---- Circumferences via silhouette + ellipse approximation ----
    # Section heights as fractions of torso, measured top-down from shoulder midpoint.
    chest_y = sh_mid[1] + 0.20 * (hip_mid[1] - sh_mid[1])
    waist_y = sh_mid[1] + 0.65 * (hip_mid[1] - sh_mid[1])
    hip_y   = hip_mid[1]
    x_center = int(sh_mid[0])

    def _section_circ(y: float, section: str) -> float:
        width_px = _horizontal_silhouette_width(image_bgr, int(y), x_center)
        if width_px < 0:                                    # fallback to landmarks
            if section == 'chest':
                width_px = _dist_px(l_sh, r_sh) * 1.05
            elif section == 'waist':
                width_px = _dist_px(l_sh, r_sh) * 0.85
            else:                                           # hip
                width_px = _dist_px(l_hip, r_hip) * 1.10
            notes.append(f'{section}: silhouette extraction failed, used landmark fallback')
        width_cm = width_px * cm_per_px
        depth_cm = width_cm * DEPTH_WIDTH_RATIO[section][gender]
        return ellipse_perimeter(width_cm, depth_cm)

    chest_c = _section_circ(chest_y, 'chest')
    waist_c = _section_circ(waist_y, 'waist')
    hip_c   = _section_circ(hip_y,   'hip')

    # ---- Confidence assessment ----
    avg_vis = float(pose_res.visibility[[
        LM.LEFT_SHOULDER.value,  LM.RIGHT_SHOULDER.value,
        LM.LEFT_HIP.value,       LM.RIGHT_HIP.value,
        LM.LEFT_ANKLE.value,     LM.RIGHT_ANKLE.value,
    ]].mean())
    confidence = 'high' if avg_vis > 0.85 else 'medium' if avg_vis > 0.65 else 'low'
    if confidence != 'high':
        notes.append(f'Average landmark visibility: {avg_vis:.2f}. Try better lighting / tighter clothes.')

    return Measurements(
        height_cm         = round(user_height_cm,    1),
        shoulder_width_cm = round(shoulder_width_cm, 1),
        chest_circ_cm     = round(chest_c,           1),
        waist_circ_cm     = round(waist_c,           1),
        hip_circ_cm       = round(hip_c,             1),
        sleeve_length_cm  = round(sleeve_length_cm,  1),
        arm_length_cm     = round(arm_length_cm,     1),
        torso_length_cm   = round(torso_length_cm,   1),
        inseam_cm         = round(inseam_cm,         1),
        neck_to_waist_cm  = round(neck_to_waist_cm,  1),
        cm_per_pixel      = round(cm_per_px,         4),
        confidence        = confidence,
        notes             = notes,
    )

## 5. Size recommendation engine

Maps measurements to size charts for kurta, kurti, and saree blouse. Returns size + the **reasoning** behind the choice — that explainability is the actual product moat.

In [ ]:
# Standard Indian size charts. Values are inclusive ranges in cm.
# Source: averaged across Fabindia, Manyavar, BIBA published charts.
SIZE_CHARTS = {
    'mens_kurta': {
        'XS': {'chest': (84,  89),  'length': (98, 102)},
        'S':  {'chest': (89,  94),  'length': (100, 104)},
        'M':  {'chest': (94,  99),  'length': (102, 106)},
        'L':  {'chest': (99,  104), 'length': (104, 108)},
        'XL': {'chest': (104, 109), 'length': (106, 110)},
        'XXL':{'chest': (109, 116), 'length': (108, 112)},
    },
    'womens_kurta': {
        'XS': {'chest': (78,  83),  'waist': (62, 67),  'hip': (86, 91)},
        'S':  {'chest': (83,  88),  'waist': (67, 72),  'hip': (91, 96)},
        'M':  {'chest': (88,  93),  'waist': (72, 77),  'hip': (96, 101)},
        'L':  {'chest': (93,  98),  'waist': (77, 82),  'hip': (101, 106)},
        'XL': {'chest': (98,  104), 'waist': (82, 88),  'hip': (106, 112)},
        'XXL':{'chest': (104, 110), 'waist': (88, 96),  'hip': (112, 118)},
    },
    'saree_blouse': {
        '32': {'chest': (80,  82)},
        '34': {'chest': (84,  86)},
        '36': {'chest': (88,  90)},
        '38': {'chest': (92,  94)},
        '40': {'chest': (96,  98)},
        '42': {'chest': (100, 102)},
        '44': {'chest': (104, 106)},
    },
}


@dataclass
class SizeRecommendation:
    garment:        str
    recommended:    str
    confidence:     Literal['high', 'medium', 'low']
    reasoning:      list[str]
    fit_notes:      list[str]


def _score_size(measurement: float, size_range: tuple[float, float]) -> float:
    """0.0 = perfect center fit, larger = farther from range."""
    lo, hi = size_range
    if lo <= measurement <= hi:
        center = (lo + hi) / 2.0
        return abs(measurement - center) / (hi - lo)
    return min(abs(measurement - lo), abs(measurement - hi)) + 1.0


def recommend_size(m: Measurements, garment: str) -> SizeRecommendation:
    """Pick the best size by minimizing total deviation across all measured dims."""
    if garment not in SIZE_CHARTS:
        raise ValueError(f'Unknown garment: {garment}. Choose from {list(SIZE_CHARTS)}')
    chart = SIZE_CHARTS[garment]

    user_dims = {
        'chest':  m.chest_circ_cm,
        'waist':  m.waist_circ_cm,
        'hip':    m.hip_circ_cm,
        'length': m.neck_to_waist_cm * 1.6,         # rough kurta length proxy
    }

    scores: dict[str, float] = {}
    breakdowns: dict[str, list[str]] = {}
    for size, ranges in chart.items():
        total = 0.0
        notes: list[str] = []
        for dim, rng in ranges.items():
            user_val = user_dims[dim]
            sc = _score_size(user_val, rng)
            total += sc
            in_range = rng[0] <= user_val <= rng[1]
            mark = 'in range' if in_range else f'outside (range {rng[0]}–{rng[1]})'
            notes.append(f'{dim} {user_val:.1f}cm: {mark}')
        scores[size] = total
        breakdowns[size] = notes

    best = min(scores, key=scores.get)
    best_score = scores[best]

    # Confidence: how decisive is the winner?
    sorted_scores = sorted(scores.values())
    margin = sorted_scores[1] - sorted_scores[0] if len(sorted_scores) > 1 else 1.0
    if best_score < 0.4 and margin > 0.3:
        confidence = 'high'
    elif best_score < 1.2:
        confidence = 'medium'
    else:
        confidence = 'low'

    fit_notes: list[str] = []
    chart_best = chart[best]
    for dim, rng in chart_best.items():
        user_val = user_dims[dim]
        if user_val < rng[0]:
            fit_notes.append(f'{dim.capitalize()} will be loose by ~{rng[0] - user_val:.1f}cm.')
        elif user_val > rng[1]:
            fit_notes.append(f'{dim.capitalize()} will be tight by ~{user_val - rng[1]:.1f}cm.')
        else:
            fit_notes.append(f'{dim.capitalize()} fits comfortably.')

    return SizeRecommendation(
        garment     = garment,
        recommended = best,
        confidence  = confidence,
        reasoning   = breakdowns[best],
        fit_notes   = fit_notes,
    )

## 6. Quick smoke test

Run this cell with a sample image URL to verify the pipeline before launching the UI.

In [ ]:
import urllib.request

# Public domain test image of a person standing
TEST_URL = 'https://images.pexels.com/photos/1300402/pexels-photo-1300402.jpeg?w=800'
urllib.request.urlretrieve(TEST_URL, '/tmp/test.jpg')

img = cv2.imread('/tmp/test.jpg')
pose_res = estimate_pose(img)
assert pose_res is not None, 'No person detected in test image'
print('Pose detected. Full body visible:', pose_res.is_full_body_visible())

# Assume 175cm height for the test subject
m = compute_measurements(img, pose_res, user_height_cm=175.0, gender='male')
print('\nMeasurements:')
for k, v in m.to_dict().items():
    print(f'  {k}: {v}')

rec = recommend_size(m, 'mens_kurta')
print(f'\nRecommended size: {rec.recommended} (confidence: {rec.confidence})')
for n in rec.fit_notes:
    print(f'  - {n}')

## 7. Gradio web UI

Launches a public link valid for 72 hours. Share this for your demo video / LinkedIn post.

In [ ]:
import gradio as gr

GARMENT_CHOICES = {
    "Men's Kurta":   'mens_kurta',
    "Women's Kurta / Anarkali": 'womens_kurta',
    'Saree Blouse':  'saree_blouse',
}


def run_pipeline(
    image_pil:      Image.Image,
    height_cm:      float,
    gender:         str,
    garment_label:  str,
):
    if image_pil is None:
        return None, 'Upload a photo to begin.', '', ''
    if height_cm < 100 or height_cm > 230:
        return None, 'Enter a realistic height (100–230 cm).', '', ''

    img_rgb = np.array(image_pil)
    img_bgr = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2BGR)

    pose_res = estimate_pose(img_bgr)
    if pose_res is None:
        return None, 'No person detected. Use a clear front-facing full-body photo.', '', ''
    if not pose_res.is_full_body_visible():
        return None, 'Full body not visible. Show head to ankles in frame.', '', ''

    m = compute_measurements(img_bgr, pose_res, height_cm, gender=gender)
    rec = recommend_size(m, GARMENT_CHOICES[garment_label])

    annotated = draw_pose(img_bgr, pose_res)
    annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)

    measurements_md = '### Measurements\n\n' + '\n'.join([
        f'| {k.replace("_", " ").title()} | {v} |'
        for k, v in m.to_dict().items()
        if k not in ('confidence', 'notes', 'cm_per_pixel')
    ])
    measurements_md = (
        '| Measurement | Value (cm) |\n|---|---|\n' + measurements_md.split('\n\n', 1)[1]
        + f'\n\n**Pose confidence:** {m.confidence}'
    )
    if m.notes:
        measurements_md += '\n\n_Notes:_ ' + '; '.join(m.notes)

    rec_md = (
        f'### Recommended size: **{rec.recommended}**\n\n'
        f'_Confidence: {rec.confidence}_\n\n'
        f'**Reasoning:**\n' + '\n'.join(f'- {r}' for r in rec.reasoning) + '\n\n'
        f'**Fit notes:**\n' + '\n'.join(f'- {n}' for n in rec.fit_notes)
    )

    disclaimer = (
        '_Stage 1 demo: circumferences estimated from silhouette + anthropometric '
        'depth ratios. Stage 2 will use Depth Anything v2 for true 3D measurements._'
    )

    return annotated_rgb, measurements_md, rec_md, disclaimer


with gr.Blocks(title='FitMirror — Stage 1', theme=gr.themes.Soft()) as demo:
    gr.Markdown(
        '# FitMirror\n'
        '### One photo → tailoring measurements + size recommendation for Indian wear.\n'
        '_Built by Abbas | github.com/Abbas5055/FitMirror_'
    )

    with gr.Row():
        with gr.Column(scale=1):
            inp_img    = gr.Image(type='pil', label='Front-facing full-body photo')
            inp_height = gr.Number(value=170, label='Your height (cm)', precision=0)
            inp_gender = gr.Radio(['male', 'female'], value='male', label='Gender (for body proportions)')
            inp_garm   = gr.Dropdown(list(GARMENT_CHOICES.keys()), value="Men's Kurta", label='Garment')
            btn        = gr.Button('Analyze', variant='primary')
        with gr.Column(scale=1):
            out_img    = gr.Image(label='Pose detected')
            out_meas   = gr.Markdown()
            out_rec    = gr.Markdown()
            out_note   = gr.Markdown()

    btn.click(
        run_pipeline,
        inputs=[inp_img, inp_height, inp_gender, inp_garm],
        outputs=[out_img, out_meas, out_rec, out_note],
    )

    gr.Markdown(
        '### Tips for best results\n'
        '- Stand straight, arms slightly away from body\n'
        '- Wear fitted clothes (loose clothes → inflated measurements)\n'
        '- Plain background, full body in frame (head to ankles)\n'
        '- Good lighting, no strong shadows'
    )

demo.launch(share=True, debug=False)

## 8. Next steps

**After this notebook works end-to-end:**

1. **Validate accuracy** — measure yourself + 4 friends with a real tape, log errors. Build accuracy table for README.
2. **Record demo video** — 90-sec Loom: problem → upload photo → show measurements + size → close on the explainability angle.
3. **Migrate to HF Spaces** — modularize this notebook into `app.py` + `fitmirror/` package, push to a HF Space repo, get a permanent URL.
4. **Stage 2** — add Depth Anything v2 + SMPL fitting for true 3D-grounded circumferences.
5. **Stage 3** — add garment try-on rendering via TPS warp.

**Submit the Stage 1 link to the apparel role's recruiter NOW.** A working demo with sizing reasoning beats a half-built try-on system. Stage 2/3 are upgrades for after the intro call.